# Create footprint for the HJBL area

This notebook creates a footprint for the HJBL study area by combining:
1. The **Hudson Plain** ecozone polygon — defines the inland boundary
2. The [**GADM Canada**](https://gadm.org/) boundary — provides an accurate coastline

In [ ]:
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import geopandas as gpd

In [ ]:
# 1. Download GADM Canada boundaries
gadm_dir = Path("../data/raw/vectors/gadm_canada")
gadm_zip = gadm_dir / "gadm41_CAN_shp.zip"

if not gadm_dir.exists():
    gadm_dir.mkdir(parents=True)
    url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_CAN_shp.zip"
    print(f"Downloading GADM Canada → {gadm_zip}")
    urlretrieve(url, gadm_zip)
    with zipfile.ZipFile(gadm_zip, "r") as zf:
        zf.extractall(gadm_dir)
    print("Done")
else:
    print(f"Already downloaded: {gadm_dir}")

In [ ]:
# 2. Hudson Plain ecozone as the base shape
ecozones = gpd.read_file("../data/raw/vectors/ecozones/Ecozones/ecozones.shp")
hudson_plain = ecozones[ecozones["ZONE_NAME"] == "Hudson Plain"].to_crs(epsg=4326)
hudson_plain = hudson_plain.dissolve()

In [ ]:
# 3. Clip Hudson Plain to Canada boundary for an accurate coastline
canada = gpd.read_file(gadm_dir / "gadm41_CAN_0.shp").to_crs(epsg=4326)
footprint = gpd.overlay(hudson_plain, canada, how="intersection")

output_path = Path("../data/processed/vectors/HJBL_footprint.geojson")
output_path.parent.mkdir(parents=True, exist_ok=True)
footprint.to_file(output_path, driver="GeoJSON")
print(f"Footprint saved to: {output_path}")